# GOAT-TS End-to-End: Ingest → Goal-Seed → Reason → Viz

This notebook runs a minimal pipeline **without Docker**:
1. Load sample text (puzzle/corpus).
2. Run the cognition loop with **goal-directed seeding** (dry-run).
3. Run the reasoning loop on a query.
4. Inspect results and optionally open the Streamlit viz.

Run from repo root: `jupyter notebook examples/goat_end_to_end.ipynb` (or JupyterLab).

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd() if Path.cwd().name != "examples" else Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("ROOT:", ROOT)

## 1. Load sample puzzle / corpus

Use the built-in sample or point to your own text file (one chunk per line).

In [ ]:
sample_path = ROOT / "examples" / "sample_input.txt"
chunks = sample_path.read_text(encoding="utf-8").strip().splitlines()
chunks = [c.strip() for c in chunks if c.strip()]
print(f"Loaded {len(chunks)} chunks. First: {chunks[0][:60]}...")

## 2. Cognition loop with goal-directed seeding (dry-run)

Seeds from labels like `concept`; add **goal_labels** to bias activation toward e.g. `knowledge`, `graph`.

In [ ]:
from src.agi_loop.demo_loop import run_demo
from src.graph.client import NebulaGraphClient

config_path = str(ROOT / "configs" / "graph.yaml")
client = NebulaGraphClient(config_path=config_path, dry_run_override=True)

nodes, edges, summary = run_demo(
    client,
    seed_ids=[],
    seed_labels=["concept"],
    goal_labels=["knowledge", "graph"],
    ticks=5,
    verbose=False,
    config_path=config_path,
)
client.close()

print("Summary:", summary["ticks"], "ticks, seeds:", summary["seed_count"])
print("Final states:", summary.get("final_states"))
print("Node count:", len(nodes), "Edge count:", len(edges))

## 3. Reasoning loop

Run a query and get activated nodes, tension, and hypotheses.

In [ ]:
from src.reasoning.loop import run_reasoning_loop

query = "Wikipedia supports free knowledge and Wikidata supports structured facts. How are they related?"
response = run_reasoning_loop(query, ROOT, live=False)

print("Query:", response.query)
print("Tension score:", response.tension.score)
print("Graph context:", response.graph_context)
print("Hypotheses (first 3):")
for h in response.hypotheses[:3]:
    print(" -", h.prompt)

## 4. Viz and next steps

To view the **interactive Plotly/NetworkX graph**, run in a terminal from repo root:

```bash
python -m streamlit run scripts/streamlit_viz.py
```

Then choose **Interactive graph** in the sidebar. For the full GUI (Setup Wizard, Config, Ingestion, Reasoning, Export & API):

```bash
python -m streamlit run scripts/goat_ts_gui.py
```

In [ ]:
# Quick in-notebook summary of top activated nodes (from last demo run)
top = sorted(nodes, key=lambda n: n.activation, reverse=True)[:8]
for n in top:
    print(f"  {n.label[:40]:40} activation={n.activation:.3f}")